<a href="https://colab.research.google.com/github/Paulo83-dev/mestrado-computacao-aplicada/blob/main/machine-learning/aula04d - GridSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Aula 04d: Busca de Hiperparâmetros (GridSearch, RandomSearch e Optuna)

Nesta aula, estudaremos como automatizar a **busca de hiperparâmetros** para encontrar a melhor configuração para o nosso modelo de Machine Learning.

Aprofundaremos nosso conhecimento em:
1. **GridSearchCV (Busca em Grade):** Testa exaustivamente todas as combinações de parâmetros.
2. **RandomizedSearchCV (Busca Aleatória):** Sorteia combinações aleatórias de parâmetros, sendo muito mais rápido para espaços grandes.
3. **Validação Cruzada Aninhada (Nested Cross-Validation):** Para estimar sem viés a performance de generalização.
4. **Otimização com Optuna:** Uma ferramenta moderna baseada em algoritmos Bayesianos.

In [215]:
# Carrega a base Wine e divide em treino e teste
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### 📦 GridSearchCV Simples (Variando K)

#### 🔍 O que este bloco faz?
Configura um `GridSearchCV` para testar os valores de $K$ ímpares de 1 a 19 no K-NN, usando validação cruzada padrão (K-Fold com 5 splits por padrão).

#### 🎯 Qual a intenção pedagógica?
Introduzir a sintaxe básica do `GridSearchCV`. O objeto `grid` se comporta como um estimador completo: ao chamarmos `fit`, ele roda todo o processo de busca em grade e seleciona a melhor configuração (`grid.best_params_`).

In [323]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Busca em grade simples variando o K do K-NN
params = {'n_neighbors': range(1, 21, 2)}
model = KNeighborsClassifier()
grid = GridSearchCV(model, params, scoring='accuracy')
grid.fit(X_train, y_train)
print("Melhor parâmetro:", grid.best_params_)
print("Melhor score obtido:", grid.best_score_)

{'n_neighbors': 1}
0.7613300492610836


### 🏁 Avaliação no Conjunto de Teste

#### 🔍 O que este bloco faz?
Usa o melhor modelo encontrado pela busca em grade para fazer predições no conjunto de teste.

#### 🎯 Qual a intenção pedagógica?
Demonstrar que o `grid` pode ser usado diretamente para chamar o método `predict`, pois o Scikit-Learn re-treina automaticamente o modelo com a melhor combinação de parâmetros usando todo o conjunto de treino disponível.

In [324]:
# Predição no conjunto de teste usando o melhor modelo selecionado
y_pred = grid.predict(X_test)
print("Acurácia final no teste:", accuracy_score(y_test, y_pred))

0.6111111111111112


### 🎛️ GridSearch Multi-Parâmetro com KFold

#### 🔍 O que este bloco faz?
Expande a busca para múltiplos hiperparâmetros do K-NN: número de vizinhos (`n_neighbors`), função de peso (`weights`) e métrica de distância (`metric`), sob uma partição `KFold` explícita.

#### 🎯 Qual a intenção pedagógica?
Mostrar como otimizar múltiplos parâmetros conjuntamente. A busca exaustiva em grade agora testa todas as combinações ($10 \times 2 \times 3 = 60$ combinações) para cada um dos 5 folds, executando $300$ treinos no total.

In [334]:
from sklearn.model_selection import KFold

# GridSearch multi-parâmetro associado ao KFold explícito
params = {
    'n_neighbors': range(1, 21, 2),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
model = KNeighborsClassifier()
grid = GridSearchCV(model, params, scoring='accuracy', cv=KFold(n_splits=5, shuffle=True))
grid.fit(X_train, y_train)
print("Melhor parâmetro:", grid.best_params_)
print("Melhor score obtido:", grid.best_score_)

{'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
0.8032019704433498


### 🔄 GridSearch com RepeatedKFold

#### 🔍 O que este bloco faz?
Executa a mesma busca em grade utilizando a partição robusta `RepeatedKFold`.

#### 🎯 Qual a intenção pedagógica?
Garantir a estabilidade da escolha dos hiperparâmetros, reduzindo a variância causada pelo sorteio aleatório dos folds.

In [335]:
from sklearn.model_selection import RepeatedKFold

# GridSearch multi-parâmetro associado ao RepeatedKFold
model = KNeighborsClassifier()
grid = GridSearchCV(model, params, scoring='accuracy',
                    cv=RepeatedKFold(n_splits=5, n_repeats=10))
grid.fit(X_train, y_train)
print("Melhor parâmetro:", grid.best_params_)
print("Melhor score obtido:", grid.best_score_)

{'metric': 'manhattan', 'n_neighbors': 1, 'weights': 'uniform'}
0.8015270935960591


### 📉 Estimação de Performance (Baseline de Validação Cruzada)

#### 🔍 O que este bloco faz?
Mede a acurácia do K-NN padrão usando a validação cruzada (`cross_val_score`).

#### 🎯 Qual a intenção pedagógica?
Estabelecer a acurácia de referência (~68%) antes de iniciarmos a validação cruzada aninhada.

In [347]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Medindo a acurácia de generalização padrão com K-Fold
scores = cross_val_score(KNeighborsClassifier(), X_train, y_train,
                         cv=KFold(n_splits=5, shuffle=True))
print("Pontuações dos folds:", scores)
print("Média:", np.mean(scores))

[0.51724138 0.75862069 0.67857143 0.60714286 0.85714286]
0.683743842364532


### 📉 Estimação de Performance (Baseline de Validação Cruzada)

#### 🔍 O que este bloco faz?
Mede a acurácia do K-NN padrão usando a validação cruzada (`cross_val_score`).

#### 🎯 Qual a intenção pedagógica?
Estabelecer a acurácia de referência (~68%) antes de iniciarmos a validação cruzada aninhada.

In [348]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Medindo a acurácia de generalização padrão com K-Fold
scores = cross_val_score(KNeighborsClassifier(), X_train, y_train,
                         cv=KFold(n_splits=5, shuffle=True))
print("Pontuações dos folds:", scores)
print("Média:", np.mean(scores))

[0.86206897 0.75862069 0.78571429 0.82142857 0.82142857]
0.8098522167487683


### 🔗 Pipeline com GridSearch Aninhado

#### 🔍 O que este bloco faz?
Cria um pipeline contendo o `StandardScaler` e o estimador de busca em grade `grid`.

#### 🎯 Qual a intenção pedagógica?
Observe a estrutura: aqui, o scaler está fora da busca em grade. Embora funcione, isso significa que a normalização é feita na base de treino externa, e os dados do loop interno podem ter leve vazamento de informações de escala. Veremos a correção a seguir.

In [349]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Scaler externo, modelo-grid interno
pipeline = Pipeline([
  ('scaler', StandardScaler()),
  ('model', grid)
])
scores = cross_val_score(pipeline, X_train, y_train, cv=KFold(n_splits=5, shuffle=True))
print("Pontuações:", scores)
print("Média:", np.mean(scores))

[1.         0.96551724 0.96428571 0.96428571 0.89285714]
0.9573891625615765


### 📉 Estimação de Performance (Baseline de Validação Cruzada)

#### 🔍 O que este bloco faz?
Mede a acurácia do K-NN padrão usando a validação cruzada (`cross_val_score`).

#### 🎯 Qual a intenção pedagógica?
Estabelecer a acurácia de referência (~68%) antes de iniciarmos a validação cruzada aninhada.

In [351]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Medindo a acurácia de generalização padrão com K-Fold
scores = cross_val_score(KNeighborsClassifier(), X_train, y_train,
                         cv=KFold(n_splits=5, shuffle=True))
print("Pontuações dos folds:", scores)
print("Média:", np.mean(scores))

[0.96551724 1.         0.96428571 0.96428571 0.96428571]
0.9716748768472907


### 📉 Estimação de Performance (Baseline de Validação Cruzada)

#### 🔍 O que este bloco faz?
Mede a acurácia do K-NN padrão usando a validação cruzada (`cross_val_score`).

#### 🎯 Qual a intenção pedagógica?
Estabelecer a acurácia de referência (~68%) antes de iniciarmos a validação cruzada aninhada.

In [352]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Medindo a acurácia de generalização padrão com K-Fold
scores = cross_val_score(KNeighborsClassifier(), X_train, y_train,
                         cv=KFold(n_splits=5, shuffle=True))
print("Pontuações dos folds:", scores)
print("Média:", np.mean(scores))

[1.         0.93103448 1.         1.         0.96428571]
0.9790640394088669


### 🔄 Validação Cruzada Aninhada (Nested Cross-Validation)

#### 🔍 O que este bloco faz?
Executa `cross_val_score` passando o objeto `grid` (GridSearchCV) como o estimador principal.

#### 🎯 Qual a intenção pedagógica?
**Este é o padrão-ouro para validação em Machine Learning:**
1. **Loop Externo:** Divide os dados em treino e teste para estimar a acurácia geral do modelo.
2. **Loop Interno:** O `GridSearchCV` subdivide a base de treino externa para escolher os melhores hiperparâmetros.
Isso impede que o vazamento de informações do ajuste de parâmetros contamine a estimativa de performance final.

In [359]:
params = {
    'n_neighbors': range(1, 21, 2),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
model = KNeighborsClassifier()
grid = GridSearchCV(model, params, scoring='accuracy', cv=KFold(n_splits=5, shuffle=True))

# Validação Cruzada Aninhada (Nested CV)
scores = cross_val_score(grid, X_train, y_train, cv=KFold(n_splits=5, shuffle=True))
print("Pontuações aninhadas:", scores)
print("Média aninhada:", np.mean(scores))

[1.         0.96551724 0.92857143 1.         0.89285714]
0.9573891625615765


### 📦 Instalando o Optuna

#### 🔍 O que este bloco faz?
Instala a biblioteca **Optuna**, um framework moderno de otimização de hiperparâmetros.

In [361]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.5 MB/s eta 0:00:00


### 🚀 Otimização Bayesiana com Optuna

#### 🔍 O que este bloco faz?
Define uma função objetivo que recebe uma sugestão de parâmetros (`trial`), constrói o pipeline e retorna a média da validação cruzada. Em seguida, inicia o estudo para maximizar essa métrica.

#### 🎯 Qual a intenção pedagógica?
Apresentar a Otimização Bayesiana. Diferente do RandomSearch, o Optuna aprende com as tentativas anteriores (usando algoritmos como TPE - Tree-structured Parzen Estimator) para focar a busca nas regiões mais promissoras do espaço de busca, sendo extremamente eficiente para problemas complexos.

In [364]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold, cross_val_score
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Função objetivo para o Optuna otimizar
def objective(trial):
    # Parâmetros sugeridos pelo Optuna para o scaler
    with_mean = trial.suggest_categorical('scaler__with_mean', [True, False])
    with_std = trial.suggest_categorical('scaler__with_std', [True, False])

    # Parâmetros sugeridos pelo Optuna para o classificador K-NN
    n_neighbors = trial.suggest_int('model__n_neighbors', 1, 19, step=2)
    weights = trial.suggest_categorical('model__weights', ['uniform', 'distance'])
    metric = trial.suggest_categorical('model__metric', ['euclidean', 'manhattan', 'minkowski'])

    pipeline = Pipeline([
        ('scaler', StandardScaler(with_mean=with_mean, with_std=with_std)),
        ('model', KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric))
    ])

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
# Configura logs mais silenciosos do Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=15)

print("Número de experimentos concluídos:", len(study.trials))
print("Melhor experimento:")
trial = study.best_trial
print("  Acurácia obtida:", trial.value)
print("  Parâmetros recomendados:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-08 00:47:33,008] A new study created in memory with name: no-name-8f4b3baa-09c6-4da7-8eac-0841f66dad66
/tmp/ipykernel_18898/2138879417.py:14: UserWarning: The distribution is specified by [1, 20] and step=2, but the range is not divisible by `step`. It will be replaced with [1, 19].
  n_neighbors = trial.suggest_int('model__n_neighbors', 1, 20, step=2)
[I 2026-04-08 00:47:33,038] Trial 0 finished with value: 0.8100985221674877 and parameters: {'scaler__with_mean': False, 'scaler__with_std': False, 'model__n_neighbors': 1, 'model__weights': 'uniform', 'model__metric': 'manhattan'}. Best is trial 0 with value: 0.8100985221674877.
/tmp/ipykernel_18898/2138879417.py:14: UserWarning: The distribution is specified by [1, 20] and step=2, but the range is not divisible by `step`. It will be replaced with [1, 19].
  n_neighbors = trial.suggest_int('model__n_neighbors', 1, 20, step=2)
[I 2026-04-08 00:47:33,066] Trial 1 finished with value: 0.7753694581280788 and parameters: {'scaler_

Number of finished trials:  10
Best trial:
  Value:  0.9788177339901478
  Params: 
    scaler__with_mean: True
    scaler__with_std: True
    model__n_neighbors: 5
    model__weights: uniform
    model__metric: euclidean
